# Saudi Aviation Data: Profiling & Feature Analysis

**Source:** OurAirports open data (airports, runways, navaids, frequencies, regions, countries)
**Scope:** Saudi Arabia (`iso_country == "SA"`)

**overview :**
 This notebook profiles OurAirports data, focusing on Saudi Arabia, to assess data quality, validate relationships, analyze coverage, and define cleaning decisions for the next phase.

# 1. Imports & Dataset Links

In [76]:
import pandas as pd

BASE = "https://davidmegginson.github.io/ourairports-data/"

# keep_default_na=False: so the text "NA" (North America / Namibia) is not read as missing
read = lambda f: pd.read_csv(BASE + f, keep_default_na=False, na_values=[""])

airports    = read("airports.csv")
regions     = read("regions.csv")
runways     = read("runways.csv")
navaids     = read("navaids.csv")
frequencies = read("airport-frequencies.csv")
countries   = read("countries.csv")

# 2. Raw Dataset Overview (Global Airports)

In [77]:
airports.head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NA,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NA,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NA,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NA,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NA,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN


**Shape & Info**

In [78]:
airports.shape
airports.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86231 entries, 0 to 86230
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 86231 non-null  int64  
 1   ident              86231 non-null  object 
 2   type               86231 non-null  object 
 3   name               86231 non-null  object 
 4   latitude_deg       86231 non-null  float64
 5   longitude_deg      86231 non-null  float64
 6   elevation_ft       71261 non-null  float64
 7   continent          86231 non-null  object 
 8   iso_country        86231 non-null  object 
 9   iso_region         86231 non-null  object 
 10  municipality       81463 non-null  object 
 11  scheduled_service  86231 non-null  object 
 12  icao_code          10531 non-null  object 
 13  iata_code          9051 non-null   object 
 14  gps_code           44481 non-null  object 
 15  local_code         36125 non-null  object 
 16  home_link          477

**Types**

In [79]:
categorical_cols = [
    "type",
    "continent",
    "iso_country",
    "scheduled_service"
]

for col in categorical_cols:
    print(f"\n===== {col} =====")
    print(airports[col].value_counts(dropna=False).head(20))


===== type =====
type
small_airport     42803
heliport          23238
closed            13574
medium_airport     4105
seaplane_base      1273
large_airport      1175
balloonport          63
Name: count, dtype: int64

===== continent =====
continent
NA    39825
AS    12822
EU    12624
SA    12418
OC     4249
AF     4247
AN       46
Name: count, dtype: int64

===== iso_country =====
iso_country
US    32694
BR     8092
JP     3747
CA     3369
AU     2795
MX     2695
RU     1839
FR     1786
GB     1605
DE     1435
KR     1427
IT     1158
AR     1009
VE      899
PH      856
CN      785
CO      738
ID      731
ZA      685
IN      653
Name: count, dtype: int64

===== scheduled_service =====
scheduled_service
no     81896
yes     4335
Name: count, dtype: int64


**Numerical columns**

In [80]:
numeric_cols = [
    "latitude_deg",
    "longitude_deg",
    "elevation_ft"
]

airports[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
latitude_deg,86231.0,25.797029,26.261086,-90.000000,10.879885,35.161742,43.131745,83.091029
longitude_deg,86231.0,-27.340117,85.131764,-179.876999,-92.717449,-63.174702,23.689701,179.975700
elevation_ft,71261.0,1291.191156,1655.220376,-1266.000000,208.000000,728.000000,1607.000000,17372.000000


**Identifier **

In [81]:
identifier_cols = [
    "icao_code",
    "iata_code",
    "gps_code",
    "local_code"
]

for col in identifier_cols:
    non_null = airports[col].dropna()

    print(f"\n===== {col} =====")
    print("Non-null:", len(non_null))
    print("Unique:", non_null.nunique())
    print("Duplicate non-null:", non_null.duplicated().sum())


===== icao_code =====
Non-null: 10531
Unique: 10531
Duplicate non-null: 0

===== iata_code =====
Non-null: 9051
Unique: 9051
Duplicate non-null: 0

===== gps_code =====
Non-null: 44481
Unique: 44481
Duplicate non-null: 0

===== local_code =====
Non-null: 36125
Unique: 34625
Duplicate non-null: 1500


**Missing Counts**

In [82]:
missing = pd.DataFrame({
    "missing_count": airports.isna().sum(),
    "missing_percentage": (airports.isna().mean() * 100).round(2),
}).sort_values("missing_percentage", ascending=False)

missing

,missing_count,missing_percentage
home_link,81460,94.47
iata_code,77180,89.50
icao_code,75700,87.79
wikipedia_link,69469,80.56
keywords,64428,74.72
local_code,50106,58.11
gps_code,41750,48.42
elevation_ft,14970,17.36
municipality,4768,5.53
id,0,0.00


**Findings: global airports**
- 86,231 records, 19 columns; `id` and `ident` are complete and unique (usable as keys).
- `continent` and `iso_country` are complete. The literal text `NA` (North America / Namibia) must be read with `keep_default_na=False`, otherwise pandas silently converts it to a missing value.
- `icao_code`, `iata_code` and `gps_code` are unique where present, but `local_code` has **1,500 duplicates**, so it must not be used as a key.
- Link/description columns (`home_link`, `wikipedia_link`, `keywords`) are 75-94% empty, they are informational only.
- Coordinates are all within valid global ranges. `elevation_ft` ranges from -1,266 to 17,372 ft (negative values are plausible for sites below sea level) and is the largest missing numeric column (~17%).

# 3. Saudi Airports

**Loading dataset**

In [83]:
saudi_airports = airports[airports["iso_country"] == "SA"].copy()

print("Saudi records:", len(saudi_airports))
saudi_airports.head()

Saudi records: 193


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
9571,42257,AE-0001,small_airport,Batha Airport,24.214693,51.449890,257.0,AS,SA,SA-04,Batha,yes,OEBT,NaN,OEBT,NaN,NaN,NaN,"Ras Sumeira UAE,"
51767,5752,OE43,small_airport,Arfa Airport,21.323601,41.198299,4155.0,AS,SA,SA-02,Ta'if,no,NaN,NaN,OE43,OE43,NaN,NaN,NaN
51768,5753,OE45,small_airport,Aramco Ras Khafji Airport,28.389900,48.516499,50.0,AS,SA,SA-04,Khafji,no,NaN,NaN,OE45,OE45,NaN,NaN,NaN
51769,5754,OE46,small_airport,Al Lidem Airport,20.475901,44.757500,2225.0,AS,SA,SA-01,NaN,no,NaN,NaN,OE46,OE46,NaN,NaN,NaN
51770,42263,OE47,small_airport,Al Kharj Airport,24.060722,47.410805,1500.0,AS,SA,SA-01,Al Kharj,no,OEKJ,NaN,OEKJ,NaN,NaN,NaN,OE47


**Airport types & scheduled service**

In [84]:
print(saudi_airports["type"].value_counts(), "\n")
print(saudi_airports["scheduled_service"].value_counts())

type
heliport          91
small_airport     49
medium_airport    23
large_airport     17
closed            13
Name: count, dtype: int64 

scheduled_service
no     163
yes     30
Name: count, dtype: int64


**Region distribution (with region names)**

In [85]:
saudi_regions = regions[regions["iso_country"] == "SA"].copy()

region_counts = (
    saudi_airports["iso_region"].value_counts()
    .rename_axis("code").reset_index(name="airports")
    .merge(saudi_regions[["code", "name"]], on="code", how="left")
)
region_counts

,code,airports,name
0,SA-04,60,Ash Sharqiyah Region
1,SA-01,50,Riyadh Region
2,SA-07,23,Tabuk Region
3,SA-02,22,Makkah Region
4,SA-14,7,Asir Region
5,SA-03,7,Al Madinah Region
6,SA-09,7,Jlzan Region
7,SA-10,5,Najran Region
8,SA-05,4,Al Qasim Region
9,SA-08,4,The Northern Border Region


**Primary key validation**

In [86]:
for col in ["id", "ident"]:
    print(f"\n===== {col} =====")
    print("Missing values:", saudi_airports[col].isna().sum())
    print("Duplicate values:", saudi_airports[col].duplicated().sum())
    print("Unique values:", saudi_airports[col].nunique())
    print("Total records:", len(saudi_airports))



===== id =====
Missing values: 0
Duplicate values: 0
Unique values: 193
Total records: 193

===== ident =====
Missing values: 0
Duplicate values: 0
Unique values: 193
Total records: 193


**Missing values (Saudi airports)**

In [87]:
missing_saudi = pd.DataFrame({
    "missing_count": saudi_airports.isna().sum(),
    "missing_percentage": (
        saudi_airports.isna().mean() * 100
    ).round(2)
})

missing_saudi = missing_saudi[
    missing_saudi["missing_count"] > 0
].sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_saudi)


,missing_count,missing_percentage
local_code,184,95.34
home_link,181,93.78
iata_code,159,82.38
keywords,157,81.35
wikipedia_link,137,70.98
icao_code,132,68.39
gps_code,115,59.59
elevation_ft,65,33.68
municipality,29,15.03


**Region relationship validation** (every airport `iso_region` must exist in `regions.code`)

In [88]:
unmatched_regions = saudi_airports[~saudi_airports["iso_region"].isin(saudi_regions["code"])]

print("Saudi region records:", len(saudi_regions))
print("Distinct region codes used by airports:", saudi_airports["iso_region"].nunique())
print("Airports with unmatched region code:", len(unmatched_regions))

unused = set(saudi_regions["code"]) - set(saudi_airports["iso_region"])
print("Regions with no airport:", unused)

Saudi region records: 14
Distinct region codes used by airports: 13
Airports with unmatched region code: 0
Regions with no airport: {'SA-U-A'}


**Coordinate validation** (approximate Saudi bounding box)

In [89]:
LAT_MIN, LAT_MAX = 16, 33
LON_MIN, LON_MAX = 34.5, 55.7

outside_bounds = saudi_airports[
    ~saudi_airports["latitude_deg"].between(LAT_MIN, LAT_MAX) |
    ~saudi_airports["longitude_deg"].between(LON_MIN, LON_MAX)
]

print("Total Saudi airports:", len(saudi_airports))
print("Missing coordinates:", saudi_airports[["latitude_deg", "longitude_deg"]].isna().any(axis=1).sum())
print("Outside approximate Saudi bounds:", len(outside_bounds))
outside_bounds[["ident", "name", "latitude_deg", "longitude_deg"]]

Total Saudi airports: 193
Missing coordinates: 0
Outside approximate Saudi bounds: 0


,ident,name,latitude_deg,longitude_deg


**Suspicious record check**

A bounding box is a weak test near the borders. A second, independent check: Saudi airports normally use an
`OExx` ICAO-style ident or an `SA-xxxx` ident. Anything else (for example an `AE-` prefix, which is the UAE) is worth a manual look.

In [90]:
ident_ok = saudi_airports["ident"].str.match(r"^(OE[A-Z0-9]{2}|SA-\d+)$")

suspect = saudi_airports[~ident_ok]
print("Records with a non-Saudi ident pattern:", len(suspect))
suspect[["ident", "name", "type", "latitude_deg", "longitude_deg", "municipality", "keywords"]]

Records with a non-Saudi ident pattern: 2


,ident,name,type,latitude_deg,longitude_deg,municipality,keywords
9571,AE-0001,Batha Airport,small_airport,24.214693,51.449890,Batha,"Ras Sumeira UAE,"
55737,RSI,Red Sea International Airport,large_airport,25.627975,37.088914,Hanak,RSI


**Active / non-active flags**

Create flags to distinguish active airports from closed airports and heliports for more meaningful coverage analysis.

In [91]:
saudi_airports["is_closed"]   = saudi_airports["type"].eq("closed")
saudi_airports["is_heliport"] = saudi_airports["type"].eq("heliport")
saudi_airports["is_core"]     = ~(saudi_airports["is_closed"] | saudi_airports["is_heliport"])

saudi_airports[["is_closed", "is_heliport", "is_core"]].sum()

,0
is_closed,13
is_heliport,91
is_core,89


**Findings: Saudi airports**
- 193 records: 91 heliports, 49 small, 23 medium, 17 large, 13 closed. Only 30 have scheduled service.
- Keys (`id`, `ident`) are clean, and all region codes match the regions table.
- Coordinates are complete and inside the bounding box. The ident-pattern check flags 2 records: `AE-0001` ("Batha Airport", UAE-style ident, "Ras Sumeira UAE" in keywords) is **suspicious** and needs manual review, while `RSI` (Red Sea International Airport) is a **false positive**: a valid Saudi airport whose ident is an IATA-style code.
- The regions table has 14 Saudi region codes but airports use only 13; the unused code `SA-U-A` is likely a placeholder for unassigned records (to confirm in the regions table).
- `elevation_ft` is missing for ~34% of the records (vs 17% globally), and `municipality` for 15%.

# 4. Saudi Runways

In [92]:
saudi_runways = runways[runways["airport_ident"].isin(saudi_airports["ident"])].copy()

print("Total global runways:", len(runways))
print("Saudi runway records:", len(saudi_runways))
print("Saudi airports with runways:", saudi_runways["airport_ident"].nunique())
print("Saudi airports without runways:", len(set(saudi_airports["ident"]) - set(saudi_runways["airport_ident"])))

Total global runways: 48336
Saudi runway records: 95
Saudi airports with runways: 76
Saudi airports without runways: 117


**Keys & missing values**

In [93]:
print("Missing IDs:", saudi_runways["id"].isna().sum())
print("Duplicate IDs:", saudi_runways["id"].duplicated().sum())

missing_runways = pd.DataFrame({
    "missing_count": saudi_runways.isna().sum(),
    "missing_percentage": (saudi_runways.isna().mean() * 100).round(2),
})
missing_runways[missing_runways["missing_count"] > 0].sort_values("missing_percentage", ascending=False)

Missing IDs: 0
Duplicate IDs: 0


,missing_count,missing_percentage
he_displaced_threshold_ft,90,94.74
le_displaced_threshold_ft,88,92.63
le_elevation_ft,33,34.74
he_elevation_ft,33,34.74
le_heading_degT,16,16.84
le_latitude_deg,16,16.84
le_longitude_deg,16,16.84
he_longitude_deg,16,16.84
he_latitude_deg,16,16.84
he_heading_degT,16,16.84


**Dimension validation**

In [94]:
display(saudi_runways[["length_ft", "width_ft"]].describe())

print("Length <= 0:", (saudi_runways["length_ft"] <= 0).sum(),
      "| Width <= 0:", (saudi_runways["width_ft"] <= 0).sum())

print("\nLongest runways:")
display(
    saudi_runways[["airport_ident", "le_ident", "he_ident", "length_ft", "width_ft", "surface"]]
    .sort_values("length_ft", ascending=False).head(10)
)

,length_ft,width_ft
count,89.000000,85.000000
mean,9555.629213,137.376471
std,2665.702216,33.786788
min,3281.000000,60.000000
25%,7531.000000,105.000000
50%,10006.000000,148.000000
75%,12005.000000,148.000000
max,14222.000000,200.000000


Length <= 0: 0 | Width <= 0: 0

Longest runways:


,airport_ident,le_ident,he_ident,length_ft,width_ft,surface
30733,OEMA,17,35,14222.0,197.0,ASP
30747,OERK,15L,33R,13796.0,197.0,ASP
30748,OERK,15R,33L,13796.0,197.0,ASP
30752,OERY,01,19,13288.0,147.0,ASP
30763,OETH,17,35,13170.0,200.0,ASP
30711,OEDF,16L,34R,13124.0,197.0,ASP
30712,OEDF,16R,34L,13124.0,197.0,ASP
30722,OEJB,17,35,13124.0,148.0,ASP
30744,OEPS,17R,35L,13123.0,148.0,ASPH
30724,OEJN,16C,34C,13123.0,197.0,ASP


**Surface standardization**: the raw column mixes several spellings of the same surface.

In [95]:
print("Raw surface values:")
print(saudi_runways["surface"].value_counts(dropna=False))

Raw surface values:
surface
ASP         75
NaN          9
CON          2
Asphalt      2
asphalt      1
ASPH         1
UNK          1
SAN          1
BIT          1
Earth        1
Concrete     1
Name: count, dtype: int64


In [96]:
surface_map = {
    "ASP": "Asphalt", "ASPH": "Asphalt", "ASPHALT": "Asphalt",
    "BIT": "Asphalt",          # bituminous: treated as asphalt (decision to confirm)
    "CON": "Concrete", "CONCRETE": "Concrete",
    "SAN": "Sand", "EARTH": "Earth", "UNK": "Unknown",
}

raw_surface = saudi_runways["surface"].str.strip().str.upper()
surface_clean = raw_surface.map(surface_map)

# Missing stays NaN ("not recorded"); a non-empty value missing from the map becomes "Other"
saudi_runways["surface_clean"] = surface_clean.where(raw_surface.isna() | surface_clean.notna(), "Other")

print("Standardized surface values:")
print(saudi_runways["surface_clean"].value_counts(dropna=False))

Standardized surface values:
surface_clean
Asphalt     80
NaN          9
Concrete     3
Unknown      1
Sand         1
Earth        1
Name: count, dtype: int64


**Duplicate Runway Check**

Check for duplicate runway records using airport identifiers and runway designators, while accounting for parallel runways.

In [97]:
real_duplicates = saudi_runways[
    saudi_runways.duplicated(subset=["airport_ident", "le_ident", "he_ident"], keep=False)
]
print("Real duplicate runway records:", len(real_duplicates))
real_duplicates[["id", "airport_ident", "le_ident", "he_ident", "length_ft", "width_ft"]]

Real duplicate runway records: 0


,id,airport_ident,le_ident,he_ident,length_ft,width_ft


**Findings: Runways**

- 95 runways across 76 airports, with clean IDs, valid dimensions, and no duplicates.
- Main missing values: displaced thresholds (93–95%), elevation (35%), coordinates/headings (17%), width (11%), and surface (9%).
- Surface values are inconsistent and standardized using `surface_clean`, preserving missing values as `NaN`.

# 5. Saudi Airport Frequencies

In [98]:
saudi_frequencies = frequencies[frequencies["airport_ident"].isin(saudi_airports["ident"])].copy()

print("Total global frequency records:", len(frequencies))
print("Saudi frequency records:", len(saudi_frequencies))
print("Airports with frequencies:", saudi_frequencies["airport_ident"].nunique())
print("Missing IDs:", saudi_frequencies["id"].isna().sum(), "| Duplicate IDs:", saudi_frequencies["id"].duplicated().sum())

print("\nFrequency types:")
print(saudi_frequencies["type"].value_counts())

print("\nMissing values:")
print(saudi_frequencies.isna().sum()[lambda s: s > 0])

Total global frequency records: 30387
Saudi frequency records: 157
Airports with frequencies: 48
Missing IDs: 0 | Duplicate IDs: 0

Frequency types:
type
CTAF     33
GND      21
TWR      19
A/G      16
APP      15
ATIS     13
EMR       9
MISC      8
POST      5
CLD       3
AFIS      3
RDO       2
PMSV      2
SMC       2
ACP       1
A/D       1
OPS       1
DEP       1
Ctrl      1
Radar     1
Name: count, dtype: int64

Missing values:
description    2
dtype: int64


# 6. Saudi Navigation Aids

In [99]:
saudi_navaids = navaids[navaids["iso_country"] == "SA"].copy()

print("Saudi navaids:", len(saudi_navaids))
saudi_navaids.head()

Saudi navaids: 54


,id,filename,ident,name,type,frequency_khz,latitude_deg,longitude_deg,elevation_ft,iso_country,dme_frequency_khz,dme_channel,dme_latitude_deg,dme_longitude_deg,dme_elevation_ft,slaved_variation_deg,magnetic_variation_deg,usageType,power,associated_airport
133,85186,Arar_VORTAC_SA,AAR,Arar,VORTAC,113300,30.908001,41.142200,1811.0,SA,113300.0,080X,NaN,NaN,NaN,2.007,3.355,BOTH,HIGH,OERR
143,85196,Abqaiq_NDB_SA,AB,Abqaiq,NDB,290,25.900000,49.583302,229.0,SA,NaN,NaN,NaN,NaN,NaN,NaN,2.157,BOTH,LOW,OEBQ
157,85210,Abha_VORTAC_SA,ABH,Abha,VORTAC,112900,18.241899,42.656898,6862.0,SA,112900.0,076X,NaN,NaN,NaN,1.007,2.111,BOTH,HIGH,OEAB
297,85349,Al_Jouf_VORTAC_SA,AJF,Al Jouf,VORTAC,117800,29.789000,40.074200,2261.0,SA,117800.0,125X,NaN,NaN,NaN,NaN,3.290,BOTH,HIGH,OESK
317,85369,Al_Kharj_VORTAC_SA,AKJ,Al Kharj,VORTAC,117300,24.068199,47.407600,1479.0,SA,117300.0,120X,NaN,NaN,NaN,NaN,2.156,BOTH,LOW,NaN


**Relationship to airports**

In [100]:
linked = saudi_navaids[saudi_navaids["associated_airport"].notna()]
matched = linked["associated_airport"].isin(saudi_airports["ident"])

print("With associated airport:", len(linked))
print("Without associated airport:", saudi_navaids["associated_airport"].isna().sum())
print("Matched to Saudi airports:", matched.sum(), "| Unmatched:", (~matched).sum())
print("Distinct airports with navaids:", linked["associated_airport"].nunique())

With associated airport: 45
Without associated airport: 9
Matched to Saudi airports: 45 | Unmatched: 0
Distinct airports with navaids: 44


**Missing values**

In [101]:
missing_navaids = pd.DataFrame({
    "missing_count": saudi_navaids.isna().sum(),
    "missing_percentage": (saudi_navaids.isna().mean() * 100).round(2),
})
missing_navaids[missing_navaids["missing_count"] > 0].sort_values("missing_percentage", ascending=False)

,missing_count,missing_percentage
dme_latitude_deg,54,100.00
dme_longitude_deg,54,100.00
dme_elevation_ft,54,100.00
slaved_variation_deg,27,50.00
dme_frequency_khz,13,24.07
dme_channel,12,22.22
associated_airport,9,16.67
elevation_ft,3,5.56


**Type, usage and DME availability**

In [102]:
print(saudi_navaids["type"].value_counts(dropna=False), "\n")
print(saudi_navaids["usageType"].value_counts(dropna=False), "\n")

saudi_navaids.groupby("type").agg(
    total_records=("id", "count"),
    dme_frequency_available=("dme_frequency_khz", "count"),
    dme_channel_available=("dme_channel", "count"),
)

type
VORTAC     27
VOR-DME    14
NDB        12
TACAN       1
Name: count, dtype: int64 

usageType
BOTH        46
TERMINAL     8
Name: count, dtype: int64 



,total_records,dme_frequency_available,dme_channel_available
type,,,
NDB,12,0,0
TACAN,1,1,1
VOR-DME,14,14,14
VORTAC,27,26,27


**VORTAC records with missing DME frequency**

In [103]:
saudi_navaids[
    (saudi_navaids["type"] == "VORTAC") & (saudi_navaids["dme_frequency_khz"].isna())
][["id", "ident", "name", "type", "frequency_khz", "dme_frequency_khz", "dme_channel", "associated_airport"]]

,id,ident,name,type,frequency_khz,dme_frequency_khz,dme_channel,associated_airport
4664,89737,KFA,Dammam,VORTAC,112300,NaN,072X,OEDF


**Findings: Navaids**

- 54 navigation aids: 27 VORTAC, 14 VOR-DME, 12 NDB, and 1 TACAN.
- 45 records are linked to 44 airports; 9 have no airport association.
- DME location and elevation fields are entirely missing, while missing DME data for NDBs is expected.

# 7. Countries & Regions Validation

In [104]:
print("Country records:", len(countries))
print("Duplicate IDs:", countries["id"].duplicated().sum(), "| Duplicate codes:", countries["code"].duplicated().sum())

saudi_country = countries[countries["code"] == "SA"]
display(saudi_country)

# Global check: every airport country / region code must exist in the reference tables
print("Airports with unmatched country code:", (~airports["iso_country"].isin(countries["code"])).sum())
print("Airports with unmatched region code :", (~airports["iso_region"].isin(regions["code"])).sum())

Country records: 249
Duplicate IDs: 0 | Duplicate codes: 0


,id,code,name,continent,wikipedia_link,keywords
190,302660,SA,Saudi Arabia,AS,https://en.wikipedia.org/wiki/Saudi_Arabia,"مطارات المملكة العربية السعودية,المطارات لموسم..."


Airports with unmatched country code: 0
Airports with unmatched region code : 0


**Findings: countries & regions**
- 249 country records, no duplicate IDs or codes; the Saudi record is complete.
- All airport country codes match the countries table (this check relies on reading `NA` correctly; Namibia's code is `NA`).
- No airport in the global table has an unmatched region code.

# 8. Cross-Dataset Integrity & Coverage

**Referential integrity.** The check is run against the **full global tables**. Checking a table that was
already filtered with `isin()` can only return 0 and proves nothing.

In [105]:
all_idents = set(airports["ident"])

orphan_runways = runways[~runways["airport_ident"].isin(all_idents)]
orphan_freqs   = frequencies[~frequencies["airport_ident"].isin(all_idents)]
orphan_navaids = navaids[navaids["associated_airport"].notna() & ~navaids["associated_airport"].isin(all_idents)]

print("Runways pointing to a non-existing airport :", len(orphan_runways))
print("Frequencies pointing to a non-existing airport:", len(orphan_freqs))
print("Navaids pointing to a non-existing airport    :", len(orphan_navaids))

Runways pointing to a non-existing airport : 0
Frequencies pointing to a non-existing airport: 0
Navaids pointing to a non-existing airport    : 0


**Coverage: airports with linked runways / frequencies / navaids**

In [106]:
coverage = saudi_airports[["ident", "name", "type", "iso_region", "is_core"]].copy()
coverage["has_runways"]     = coverage["ident"].isin(saudi_runways["airport_ident"])
coverage["has_frequencies"] = coverage["ident"].isin(saudi_frequencies["airport_ident"])
coverage["has_navaids"]     = coverage["ident"].isin(saudi_navaids["associated_airport"].dropna())

flags = ["has_runways", "has_frequencies", "has_navaids"]

def coverage_table(df, label):
    t = pd.DataFrame({"count": df[flags].sum(), "percent": (df[flags].mean() * 100).round(2)})
    t.columns = pd.MultiIndex.from_product([[label + f" (n={len(df)})"], t.columns])
    return t

pd.concat([
    coverage_table(coverage, "All records"),
    coverage_table(coverage[coverage["is_core"]], "Core airports (no heliport/closed)"),
], axis=1)

All records (n=193)          \
                              count percent   
has_runways                      76   39.38   
has_frequencies                  48   24.87   
has_navaids                      44   22.80   

                Core airports (no heliport/closed) (n=89)          
                                                    count percent  
has_runways                                            71   79.78  
has_frequencies                                        47   52.81  
has_navaids                                            43   48.31

**Coverage by airport type**

In [107]:
coverage.groupby("type")[flags].agg(["sum", "mean"]).round(2)

has_runways       has_frequencies       has_navaids      
                       sum  mean             sum  mean         sum  mean
type                                                                    
closed                   4  0.31               1  0.08           1  0.08
heliport                 1  0.01               0  0.00           0  0.00
large_airport           17  1.00              14  0.82          13  0.76
medium_airport          22  0.96              20  0.87          19  0.83
small_airport           32  0.65              13  0.27          11  0.22

**Airports with no linked infrastructure at all**

In [108]:
no_linked = coverage[~coverage[flags].any(axis=1)]
print("Airports without any linked record:", len(no_linked))
print(no_linked["type"].value_counts())

Airports without any linked record: 116
type
heliport         90
small_airport    17
closed            9
Name: count, dtype: int64


**Findings: Coverage**

- No orphan records were found in the global datasets.
- Overall coverage is low: runways (39%), frequencies (25%), and navaids (23%), largely influenced by heliports and closed airports.
- Among 89 core airports, coverage increases to 80% for runways, 53% for frequencies, and 48% for navaids.
- Large and medium airports have higher coverage, while small airports show the largest gaps.
- 116 airports have no linked infrastructure records, mostly heliports.

# 9. Data Quality Summary

In [109]:
data_quality_summary = pd.DataFrame([
    {"dataset": name, "saudi_records": len(df),
     "missing_ids": df["id"].isna().sum(), "duplicate_ids": df["id"].duplicated().sum()}
    for name, df in [("airports", saudi_airports), ("runways", saudi_runways), ("navaids", saudi_navaids),
                     ("frequencies", saudi_frequencies), ("regions", saudi_regions), ("countries", saudi_country)]
])
data_quality_summary

,dataset,saudi_records,missing_ids,duplicate_ids
0,airports,193,0,0
1,runways,95,0,0
2,navaids,54,0,0
3,frequencies,157,0,0
4,regions,14,0,0
5,countries,1,0,0


## Cleaning plan (decisions for the next phase)

| # | Issue | Decision |
|---|-------|----------|
| 1 | Suspicious record `AE-0001` (Batha) | Review manually, most likely remove from the Saudi set (`RSI` is a valid airport flagged by the ident pattern, keep it) |
| 2 | 13 closed airports | Keep but exclude from analysis through `is_closed` / `is_core` |
| 3 | 91 heliports | Keep and analyze separately for meaningful coverage comparisons. |
| 4 | `surface` spelling variants | Use `surface_clean` |
| 5 | `elevation_ft` missing (34% airports, 35% runways) | Keep as NaN; do not impute without a reliable source |
| 6 | `local_code` duplicates (global) | Do not use as a key |
| 7 | Frequencies outside VHF band | Keep; they are valid services, only re-check obvious outliers (e.g. 24.59 MHz APP) |
| 8 | Empty DME coordinate columns in navaids | Preserve in Bronze; drop entirely empty columns in Silver. |
| 9 | 9 navaids without airport | Keep; likely enroute aids (to verify) |
| 10 | Region names with source typos (`Jlzan`, `Al Batah`) | Keep as-is in bronze; correct in silver via a mapping table, keeping the original in `region_name_source` |